In [ ]:
# ---
# Get GO Terms from PlasmoDB for PmUG01-converted genes
# Input: PmUGO1_converted_genes.csv with PmUG01 IDs
# Output: all_genes_GO_terms.csv containing Gene ↔ GO mappings
# ---


In [ ]:

import json
import time
import pandas as pd
import requests

# Configuration
INPUT_FILE = "analysis_results/variants_analysis/files/PmUGO_ID_converted_genes.csv"
OUTPUT_FILE = "analysis_results/variants_analysis/files/GO_terms.csv"
BATCH_SIZE = 10
API_URL = (
    "https://plasmodb.org/plasmo/service/record-types/gene/searches/"
    "single_record_question_GeneRecordClasses_GeneRecordClass/reports/standard"
)

# Helper Functions
def load_gene_ids(input_path):
    """Load and normalize gene IDs from CSV."""
    df = pd.read_csv(input_path, sep=None, engine="python")
    df = df.replace("PMUG01", "PmUG01", regex=True)
    return df["PmUG0_ID"].dropna().unique().tolist()

def query_go_terms(gene):
    """Query GO terms for a given gene ID from PlasmoDB."""
    params = {
        "primaryKeys": f"{gene},PlasmoDB",
        "reportConfig": json.dumps({
            "attributes": ["primary_key"],
            "tables": ["GOTerms"],
            "attributeFormat": "text"
        })
    }
    try:
        response = requests.get(API_URL, params=params, timeout=10)
        if response.status_code == 200:
            records = response.json().get("records", [])
            go_terms = []
            for record in records:
                for term in record.get("tables", {}).get("GOTerms", []):
                    go_terms.append([
                        gene,
                        term.get("go_id", "N/A"),
                        term.get("go_term_name", "N/A"),
                        term.get("ontology", "N/A")
                    ])
            return go_terms
        else:
            print(f"⚠️ HTTP {response.status_code} error for {gene}")
    except requests.RequestException as e:
        print(f"❌ Request error for {gene}: {e}")
    return []

# Main Process
def fetch_all_go_terms(genes):
    go_data = []
    for i in range(0, len(genes), BATCH_SIZE):
        batch = genes[i:i + BATCH_SIZE]
        print(f"🔄 Processing batch {i // BATCH_SIZE + 1}/{-(-len(genes) // BATCH_SIZE)}...")
        for gene in batch:
            go_data.extend(query_go_terms(gene))
        time.sleep(5)  # Respect server rate limits
    return go_data

# Run Workflow
def main():
    genes = load_gene_ids(INPUT_FILE)
    go_data = fetch_all_go_terms(genes)
    df_go = pd.DataFrame(go_data, columns=["Gene", "GO_ID", "GO_Term", "Ontology"])
    df_go.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ GO terms saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()
